# **OTH Course Semantic Web & Knowledge Graphs 2026**
## **Tutorial 3: Label Property Graphs in Memgraph**

---

In this Tutorial, we will learn how to use Property Graphs in Memgraph.

Author: Sara Buchmann, sara.buchmann@oth-regensburg.de


In [2]:
!pip install -r requirements.txt


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## Memgraph Setup

Getting started: https://memgraph.com/docs/getting-started

More Information at https://memgraph.com/docs/getting-started/install-memgraph/docker


### 1. Install Docker Desktop

Memgraph runs inside Docker containers.

1. Install Docker Desktop:
    https://www.docker.com/products/docker-desktop/

2. Start Docker Desktop.
3. Verify installation in a terminal:

```bash
docker --version
docker info
```

If Docker is running, you are ready to continue.

---

### 2. Start Memgraph (Database)

Run this in a terminal or Jupyter Bash cell:

```bash
docker run -d --name -p 7687:7687 memgraph/memgraph
```

Memgraph is now running on bolt://localhost:7687


---
### 3. Start Memgraph Lab (Web UI)


```bash
docker run -p 7687:7687 -p 7444:7444 --name memgraph memgraph/memgraph-mage
```

Now open your browser: http://localhost:3000


Click **New Connection** and enter:

- Host: `host.docker.internal`
- Port: `7687`

Click **Connect**.


## Introduction to Cypher

## *How to create a Knowledge Graph*

Memgraph uses the **Cypher query language** to create and manage graph data.

### Create Nodes
A node is created using parentheses `()`.

```cypher
CREATE (:Person {name: "Anna", age: 29});
CREATE (:Person {name: "Ben", age: 33});
```

This creates a node:
- Label: Person
- Properties: name, age

### Create Relationships
Relationships are created using arrows `-[:RELATIONSHIP]->`.

```cypher
MATCH (a:Person {name: "Anna", age: 29}), (b:Person {name: "Ben", age: 33})
CREATE (a)-[:KNOWS]->(b);
```

## *How to create Cypher queries*

### Read Data
Return nodes:
```cypher
MATCH (p:Person) RETURN p;
```
Return only specific properties:
```cypher
MATCH (p:Person) RETURN p.name, p.age;
```

### Filter with WHERE:
```cypher
MATCH (p:Person) WHERE p.age > 26 RETURN p.name, p.age;
```
Multiple Conditions:
```cypher
MATCH (p:Person) WHERE p.age > 25 AND p.age < 35 RETURN p;
```
String filters:
```cypher
MATCH (p:Person) WHERE p.name STARTS WITH "A" RETURN p;
```

### Sorting with ORDER BY:
```cypher
MATCH (p:Person) RETURN p.name, p.age ORDER BY p.age;
```

### Aggregation
Count nodes:
```cypher
MATCH (p:Person) RETURN count(p);
```
Average age:
```cypher
MATCH (p:Person) RETURN avg(p.age);
```
Group results:
```cypher
MATCH (p:Person) RETURN p.age, count(*) ORDER BY p.age;
```

### LIMIT
restricts the number of returned results.
```cypher
MATCH (p:Person) RETURN p.name, p.age LIMIT 5;
```

### DISTINCT
removes duplicate results.
```cypher
MATCH (p:Person) RETURN DISTINCT p.age;
```

### MERGE
creates a node or relationship only if it does not already exist.
```cypher
MATCH (a:Person {name: "Anna"})
MERGE (a)-[:KNOWS]->(:Person {name: "Tom"});
```

### Delete Nodes
```cypher
MATCH (n) DETACH DELETE n;
```


###  Running Cypher queries in Memgraph Lab

Open your browser: http://localhost:3000

Click **New Connection** and enter your port:

- Host: `host.docker.internal`
- Port: `7687`

Click **Connect**.

In the sidebar, go to **Query execution** to run queries and go to **Graph schema** to load the graph schema.



###  Running Cypher queries in Jupyter Notebook

Install prerequisites:

```python
pip install --user pymgclient
```


Install driver:

```python
pip install gqlalchemy
```


For details see https://memgraph.github.io/gqlalchemy/

Connect to Memgraph:

```python
from gqlalchemy import Memgraph

db = Memgraph(host="127.0.0.1", port=7687)

db.execute("""CREATE (:Person {name: "Anna", age: 29})""")

list(db.execute_and_fetch("""MATCH (p:Person) RETURN p.name, p.age """))
```
Create nodes an relationships with db.execute(), Query and fetch result with db.execute_and_fetch(), use list() to get a readable representation



**NOTE** You can also use the Neo4j driver as documented here to access memgraph: https://memgraph.com/docs/client-libraries/python

## Exercise 1: Student Social Network

In this exercise you will model a **student social network** in a knowledge graph using cypher.

Each student should be represented as a node with the label **Student** and should contain the following properties:

- `name`
- `age`
- `field_of_study`

Your task is to create the students and their relationships ```cypher[:KNOWS],[:BEST_FRIEND],[:LOVES],[:HATES]``` in a graph database.

#### Students

The following students exist:

| Name   | Age | Field of Study   |
|--------|-----|------------------|
| Anna   | 22  | Data Science     |
| Ben    | 24  | Computer Science |
| Clara  | 21  | Medicine         |
| David  | 25  | Mathematics      |
| Eva    | 23  | Design           |
| Frank  | 26  | Economics        |
| Grace  | 22  | Physics          |
| Hannah | 20  | Computer Science |
| Ian    | 24  | Mathematics      |
| Julia  | 21  | Data Science     |

Relationships between Students:
- Anna is best friend with Eva, Anna knows Ben, Anna loves David
- Ben and David are best friends
- Ben loves Anna, Ben knows Eva, hates Clara
- David knows Clara, David loves Julia
- Clara ist best friend with Julia, Clara knows Anna
- Eva loves Ben, Eva knows Grace and Frank
- Frank is best friend with Grace, Frank hates Ben
- Grace loves Ben, Grace is best friend with Clara
- Hannah knows Clara, is best friend with Julia
- Ian and Frank are best friends, Ian loves Julia
- Julia is best friend with Eva and hates Anna



Tasks:
1. Find all students who are older than 24.
2. Find all students studying Computer Science.
3. Find all student Eva knows.
4. Find all students studying the same field as David.
5. Find all students who are friends of Anna’s friends.
6. Find students that are involved in love triangles.
7. Find all students who know someone that hates another student.
8. Find all students who hate someone that is loved by another student.
9.  Find students that know or are best friends with more than one person.
10. Find all students who are best friends with someone studying Data Science.
11. Find all students who love someone, but that person is hated by their own best friend



In [1]:
# Solution Exercise 1:
# docker run -d --name memgraph-social_network -p 7688:7687 memgraph/memgraph

from gqlalchemy import Memgraph

db = Memgraph(host="127.0.0.1", port=7687)

In [25]:
db.execute("""
MATCH (n)
DETACH DELETE n
""")

## Task 0: Create the knowledge graph
<details>
<summary>Show solution</summary>

```
# 0. Create the knowledge graph 
db.execute("""CREATE (:Student {name:"Anna", age:22, field_of_study:"Data Science"});""")
db.execute("""CREATE (:Student {name:"Ben", age:24, field_of_study:"Computer Science"});""")
db.execute("""CREATE (:Student {name:"Clara", age:21, field_of_study:"Medicine"});""")
db.execute("""CREATE (:Student {name:"David", age:25, field_of_study:"Mathematics"});""")
db.execute("""CREATE (:Student {name:"Eva", age:23, field_of_study:"Design"});""")
db.execute("""CREATE (:Student {name:"Frank", age:26, field_of_study:"Economics"});""")
db.execute("""CREATE (:Student {name:"Grace", age:22, field_of_study:"Physics"});""")
db.execute("""CREATE (:Student {name:"Hannah", age:20, field_of_study:"Computer Science"});""")
db.execute("""CREATE (:Student {name:"Ian", age:24, field_of_study:"Mathematics"});""")
db.execute("""CREATE (:Student {name:"Julia", age:21, field_of_study:"Data Science"});""")

# Anna is best friend with Eva
db.execute("""
MATCH (a:Student {name:"Anna"}), (e:Student {name:"Eva"})
CREATE (a)-[:BEST_FRIEND]->(e),
       (e)-[:BEST_FRIEND]->(a);
""")

# Anna knows Ben
db.execute("""
MATCH (a:Student {name:"Anna"}), (b:Student {name:"Ben"})
CREATE (a)-[:KNOWS]->(b);
""")

# Anna loves David
db.execute("""
MATCH (a:Student {name:"Anna"}), (d:Student {name:"David"})
CREATE (a)-[:LOVES]->(d);
""")

# Ben and David are best friends
db.execute("""
MATCH (b:Student {name:"Ben"}), (d:Student {name:"David"})
CREATE (b)-[:BEST_FRIEND]->(d),
       (d)-[:BEST_FRIEND]->(b);
""")

# Ben loves Anna
db.execute("""
MATCH (b:Student {name:"Ben"}), (a:Student {name:"Anna"})
CREATE (b)-[:LOVES]->(a);
""")

# Ben knows Eva
db.execute("""
MATCH (b:Student {name:"Ben"}), (e:Student {name:"Eva"})
CREATE (b)-[:KNOWS]->(e);
""")

# Ben hates Clara
db.execute("""
MATCH (b:Student {name:"Ben"}), (c:Student {name:"Clara"})
CREATE (b)-[:HATES]->(c);
""")

# David knows Clara
db.execute("""
MATCH (d:Student {name:"David"}), (c:Student {name:"Clara"})
CREATE (d)-[:KNOWS]->(c);
""")

# David loves Julia
db.execute("""
MATCH (d:Student {name:"David"}), (j:Student {name:"Julia"})
CREATE (d)-[:LOVES]->(j);
""")

# Clara is best friend with Julia
db.execute("""
MATCH (c:Student {name:"Clara"}), (j:Student {name:"Julia"})
CREATE (c)-[:BEST_FRIEND]->(j),
       (j)-[:BEST_FRIEND]->(c);
""")

# Clara knows Anna
db.execute("""
MATCH (c:Student {name:"Clara"}), (a:Student {name:"Anna"})
CREATE (c)-[:KNOWS]->(a);
""")

# Eva loves Ben
db.execute("""
MATCH (e:Student {name:"Eva"}), (b:Student {name:"Ben"})
CREATE (e)-[:LOVES]->(b);
""")

# Eva knows Grace and Frank
db.execute("""
MATCH (e:Student {name:"Eva"}), (g:Student {name:"Grace"}), (f:Student {name:"Frank"})
CREATE (e)-[:KNOWS]->(g),
       (e)-[:KNOWS]->(f);
""")

# Frank is best friend with Grace
db.execute("""
MATCH (f:Student {name:"Frank"}), (g:Student {name:"Grace"})
CREATE (f)-[:BEST_FRIEND]->(g),
       (g)-[:BEST_FRIEND]->(f);
""")

# Frank hates Ben
db.execute("""
MATCH (f:Student {name:"Frank"}), (b:Student {name:"Ben"})
CREATE (f)-[:HATES]->(b);
""")

# Grace loves Ben
db.execute("""
MATCH (g:Student {name:"Grace"}), (b:Student {name:"Ben"})
CREATE (g)-[:LOVES]->(b);
""")

# Grace is best friend with Clara
db.execute("""
MATCH (g:Student {name:"Grace"}), (c:Student {name:"Clara"})
CREATE (g)-[:BEST_FRIEND]->(c),
       (c)-[:BEST_FRIEND]->(g);
""")

# Hannah knows Clara
db.execute("""
MATCH (h:Student {name:"Hannah"}), (c:Student {name:"Clara"})
CREATE (h)-[:KNOWS]->(c);
""")

# Hannah is best friend with Julia
db.execute("""
MATCH (h:Student {name:"Hannah"}), (j:Student {name:"Julia"})
CREATE (h)-[:BEST_FRIEND]->(j),
       (j)-[:BEST_FRIEND]->(h);
""")

# Ian and Frank are best friends
db.execute("""
MATCH (i:Student {name:"Ian"}), (f:Student {name:"Frank"})
CREATE (i)-[:BEST_FRIEND]->(f),
       (f)-[:BEST_FRIEND]->(i);
""")

# Ian loves Julia
db.execute("""
MATCH (i:Student {name:"Ian"}), (j:Student {name:"Julia"})
CREATE (i)-[:LOVES]->(j);
""")

# Julia is best friend with Eva
db.execute("""
MATCH (j:Student {name:"Julia"}), (e:Student {name:"Eva"})
CREATE (j)-[:BEST_FRIEND]->(e),
       (e)-[:BEST_FRIEND]->(j);
""")

# Julia hates Anna
db.execute("""
MATCH (j:Student {name:"Julia"}), (a:Student {name:"Anna"})
CREATE (j)-[:HATES]->(a);
""")
```

</details>


## Task 1: Students older than 24
<details>
<summary>Show solution</summary>

```
# 1. Students older than 24
list (db.execute_and_fetch("""
MATCH (s:Student)
WHERE s.age > 24
RETURN s.name, s.age
"""))
```

</details>


## Task 2: Find all students studying Computer Science.
<details>
<summary>Show solution</summary>

```
# 2. Find all students studying Computer Science.
list(db.execute_and_fetch("""
MATCH (s:Student)
WHERE s.field_of_study = "Computer Science"
RETURN s.name
"""))
```

</details>


## Task 3: Find all students Eva knows.
<details>
<summary>Show solution</summary>

```
# 3. Find all students Eva knows.
list(db.execute_and_fetch("""
MATCH (:Student {name:"Eva"})-[:KNOWS]->(s)
RETURN s.name
"""))
```

</details>


## Task 4: Find all students studying the same field as David.
<details>
<summary>Show solution</summary>

```
# 4. Find all students studying the same field as David.
list(db.execute_and_fetch("""
MATCH (david:Student {name:"David"}), (s:Student)
WHERE s.field_of_study = david.field_of_study AND s <> david
RETURN s.name
"""))
```

</details>


## Task 5: Find all students who are friends of Anna’s friends.
<details>
<summary>Show solution</summary>

```
# 5. Find all students who are friends of Anna’s friends.
list(db.execute_and_fetch("""
MATCH (:Student {name:"Anna"})-[:BEST_FRIEND]->()-[:BEST_FRIEND]->(s)
WHERE s.name <> "Anna"
RETURN DISTINCT s.name
"""))
```

</details>


## Task 6: Return the names of all students involved in love triangles
<details>
<summary>Show solution</summary>

```
# 6. Return the names of all students involved in love triangles
list(db.execute_and_fetch("""
MATCH (a:Student)-[:LOVES]->(b:Student)-[:LOVES]->(c:Student)
RETURN a.name, b.name, c.name
"""))
```

</details>


## Task 7: Find all students who know someone that hates another student.
<details>
<summary>Show solution</summary>

```
#7. Find all students who know someone that hates another student.
list(db.execute_and_fetch("""
MATCH (a:Student)-[:KNOWS]->(b:Student)-[:HATES]->(c:Student)
RETURN a.name, b.name, c.name
"""))
```

</details>


## Task 8: Find all students who hate someone that is loved by another student.
<details>
<summary>Show solution</summary>

```
# 8. Find all students who hate someone that is loved by another student.
list(db.execute_and_fetch("""
MATCH (a:Student)-[:HATES]->(b:Student)<-[:LOVES]-(c:Student)
RETURN a.name, b.name, c.name
"""))
```

</details>


## Task 9: Find students that know or are best friends with more than one person.
<details>
<summary>Show solution</summary>

```
# 9. Find students that know or are best friends with more than one person.
list(db.execute_and_fetch("""
MATCH (s:Student)-[:KNOWS|BEST_FRIEND]->(p:Student)
WITH s, COUNT(p) AS connections
WHERE connections > 1
RETURN s.name, connections
"""))
```

</details>


## Task 10: Find all students who are best friends with someone studying Data Science.
<details>
<summary>Show solution</summary>

```
# 10. Find all students who are best friends with someone studying Data Science.
list(db.execute_and_fetch("""
MATCH (s:Student)-[:BEST_FRIEND]->(bf:Student)
WHERE bf.field_of_study = "Data Science"
RETURN s.name, bf.name
"""))
```

</details>


## Task 11: Find all students who love someone, but that person is hated by their own best friend
<details>
<summary>Show solution</summary>

```
# 11. Find all students who love someone, but that person is hated by their own best friend
list(db.execute_and_fetch("""
MATCH (a:Student)-[:LOVES]->(b:Student),
      (a)-[:BEST_FRIEND]->(c:Student),
      (c)-[:HATES]->(b)
RETURN a.name AS lover, b.name AS loved, c.name AS hating_best_friend
"""))
```

</details>


## Exercise 2: Tabular Data - Flight Knowledge Graph

In this exercise, we will build a **Flight Knowledge Graph** using preprocessed flight data.
The dataset contains information about flights between airports, including:

- Airlines operating the flights
- Airports (origin and destination, including country)
- Flight with arrival and departure times

The data is provided in **preprocessed CSV files** (`airports.csv`, `airlines.csv`, `flights.csv`).

**Objective:** Transform the tabular flight data into a **graph representation** to explore relationships between flights, airports, and airlines.

## Flight Knowledge Graph

We are building a **Flight Knowledge Graph** using our `airports.csv`, `airlines.csv`, and `flights.csv`.

### Nodes

| Node       | Properties |
|------------|-----------------------------|
| `Airline`  | `airline_code`, `name`  |
| `Airport`  | `airport_code`, `city`, `country` |
| `Flight`   | `flightNumber`, `origin_airport`, `destination_airport`, `Airline`,`departureTime`, `arrivalTime` |

### Relationships

| Relationship   | From → To |
|----------------|-------------------------|
| `DEPARTS_FROM` | Flight → Airport |
| `ARRIVES_AT`   | Flight → Airport |
| `OPERATED_BY`  | Flight → Airline |

---

### Steps to Build the Graph

1. Load the CSV files into dataframes.
2. **Create nodes** for **Airline**, **Airport**, and **Flight** from the dataframe columns.
2. **Add relationships**:
   - `DEPARTS_FROM`: Flight → origin airport
   - `ARRIVES_AT`: Flight → destination airport
   - `OPERATED_BY`: Flight → airline


Dont forget to start a new memgraph container:

```bash
docker run -d --name memgraph-flights -p 7689:7687 memgraph/memgraph

Tasks:
1. Is there a flight from Munich to Berlin with Lufthansa? At what time does it depart?
2. Which destinations can be reached from Munich with Lufthansa departing after 6:00 in two hours?
3. List all flights from Munich operated by Lufthansa.
4. List all flights arriving in Berlin.
5. List all flights operated by Lufthansa.
6. List all flights for any airline, including origin, destination, and times.
7. List all direct flights.
8. Which airports are served by flights from at least two different airlines?
9. List all possible flight routes between airports, including routes with one or more stopovers.
10. Find all flights operated by Lufthansa that depart before 12:00 noon
11. Which airports receive the most incoming flights?


In [ ]:
# Build Graph from the given CSV files programmatically. 
import pandas as pd

# CSVs laden
airports = pd.read_csv("data_ex21/airports.csv")
airlines = pd.read_csv("data_ex21/airlines.csv")
flights = pd.read_csv("data_ex21/flights.csv")


from gqlalchemy import Memgraph

db = Memgraph(host="127.0.0.1", port=7689)

#  Airline Nodes
for _, row in airlines.iterrows():
    db.execute(f"""
    CREATE (:Airline {{airline_code: '{row['airline_code']}', name: '{row['name']}'}})
    """)

# Airport Nodes
for _, row in airports.iterrows():
    db.execute(f"""
    CREATE (:Airport {{airport_code: '{row['airport_code']}', city: '{row['city']}', country: '{row['country']}'}})
    """)

# Flight nodes
for _, row in flights.iterrows():
    db.execute(f"""
    CREATE (:Flight {{
        flightNumber: '{row['flightNumber']}',
        origin_airport: '{row['origin_airport']}',
        destination_airport: '{row['destination_airport']}',
        Airline: '{row['Airline']}',
        departureTime: {int(row['departureTime'])},
        arrivalTime: {int(row['arrivalTime'])}
    }})
    """)

# Relationships
for _, row in flights.iterrows():
    # DEPARTS_FROM
    db.execute(f"""
    MATCH (f:Flight {{flightNumber: '{row['flightNumber']}'}}),
          (a:Airport {{airport_code: '{row['origin_airport']}'}})
    CREATE (f)-[:DEPARTS_FROM]->(a)
    """)

    # ARRIVES_AT
    db.execute(f"""
    MATCH (f:Flight {{flightNumber: '{row['flightNumber']}'}}),
          (a:Airport {{airport_code: '{row['destination_airport']}'}})
    CREATE (f)-[:ARRIVES_AT]->(a)
    """)

    # OPERATED_BY
    db.execute(f"""
    MATCH (f:Flight {{flightNumber: '{row['flightNumber']}'}}),
          (al:Airline {{name: '{row['Airline']}'}})
    CREATE (f)-[:OPERATED_BY]->(al)
    """)



## Task 1: Is there a flight from Munich to Berlin with Lufthansa? At what time does it depart?
<details>
<summary>Show solution</summary>

```
# 1. Is there a flight from Munich to Berlin with Lufthansa? At what time does it depart?
list(db.execute_and_fetch("""
MATCH (f:Flight)-[:DEPARTS_FROM]->(o:Airport {city:'Muenchen'}),
      (f)-[:ARRIVES_AT]->(d:Airport {city:'Berlin'}),
      (f)-[:OPERATED_BY]->(al:Airline {name:'Lufthansa'})
RETURN f.flightNumber
"""))
```

</details>


## Task 2: Which destinations can be reached from Munich with Lufthansa departing after 6:00 in two hours?
<details>
<summary>Show solution</summary>

```
# 2. Which destinations can be reached from Munich with Lufthansa departing after 6:00 in two hours?
list(db.execute_and_fetch("""
MATCH (f:Flight)-[:DEPARTS_FROM]->(o:Airport {city:'Muenchen'}),
      (f)-[:ARRIVES_AT]->(d:Airport),
      (f)-[:OPERATED_BY]->(al:Airline {name:'Lufthansa'})
WHERE f.departureTime > 600 AND f.arrivalTime < 800
RETURN d.city AS destination
"""))
```

</details>


## Task 3: List all flights from Munich operated by Lufthansa
<details>
<summary>Show solution</summary>

```
# 3. List all flights from Munich operated by Lufthansa
list(db.execute_and_fetch("""
MATCH (f:Flight)-[:DEPARTS_FROM]->(o:Airport {city:'Muenchen'}),
      (f)-[:ARRIVES_AT]->(d:Airport),
      (f)-[:OPERATED_BY]->(al:Airline {name:'Lufthansa'})
RETURN d.city AS destination, f.departureTime, f.arrivalTime
"""))
```

</details>


## Task 4: List all flights arriving in Berlin
<details>
<summary>Show solution</summary>

```
# 4. List all flights arriving in Berlin
list(db.execute_and_fetch("""
MATCH (f:Flight)-[:DEPARTS_FROM]->(o:Airport),
      (f)-[:ARRIVES_AT]->(d:Airport {city:'Berlin'}),
      (f)-[:OPERATED_BY]->(al:Airline)
RETURN o.city AS origin, al.name AS airline, f.arrivalTime AS arrival
"""))
```

</details>


## Task 5: List all flights operated by Lufthansa
<details>
<summary>Show solution</summary>

```
# 5. List all flights operated by Lufthansa
list(db.execute_and_fetch("""
MATCH (f:Flight)-[:DEPARTS_FROM]->(o:Airport),
      (f)-[:ARRIVES_AT]->(d:Airport),
      (f)-[:OPERATED_BY]->(al:Airline {name:'Lufthansa'})
RETURN o.city AS origin, d.city AS destination, f.departureTime, f.arrivalTime
"""))
```

</details>


## Task 6: List all flights for any airline, including origin, destination, and times
<details>
<summary>Show solution</summary>

```
# 6. List all flights for any airline, including origin, destination, and times
list(db.execute_and_fetch("""
MATCH (f:Flight)-[:DEPARTS_FROM]->(o:Airport),
      (f)-[:ARRIVES_AT]->(d:Airport),
      (f)-[:OPERATED_BY]->(al:Airline)
RETURN o.city AS origin, d.city AS destination, al.name AS airline, f.departureTime, f.arrivalTime
"""))
```

</details>


## Task 7: List all direct flights
<details>
<summary>Show solution</summary>

```
# 7. List all direct flights
list(db.execute_and_fetch("""
MATCH (f:Flight)-[:DEPARTS_FROM]->(o:Airport),
      (f)-[:ARRIVES_AT]->(d:Airport)
RETURN o.city AS origin, d.city AS destination, f.departureTime, f.arrivalTime
"""))
```

</details>


## Task 8: Which airports are served by flights from at least two different airlines?
<details>
<summary>Show solution</summary>

```
# 8. Which airports are served by flights from at least two different airlines?
list(db.execute_and_fetch("""
MATCH (f:Flight)-[:ARRIVES_AT]->(d:Airport),
      (f)-[:OPERATED_BY]->(al:Airline)
WITH d.city AS airport, COLLECT(DISTINCT al.name) AS airlines
WHERE SIZE(airlines) >= 2
RETURN airport, airlines
"""))
```

</details>


## Task 9: List all possible flight routes between airports, including routes with one or more stopovers
<details>
<summary>Show solution</summary>

```
# 9. List all possible flight routes between airports, including routes with one or more stopovers
list(db.execute_and_fetch("""
MATCH p=(f1:Flight)-[:ARRIVES_AT]->(:Airport)<-[:DEPARTS_FROM*0..]->(f2:Flight)
RETURN [n IN nodes(p) | n.flightNumber] AS route
"""))
```

</details>


## Task 10: Find all flights operated by Lufthansa that depart before 12:00 noon
<details>
<summary>Show solution</summary>

```
# 10. Find all flights operated by Lufthansa that depart before 12:00 noon
list(db.execute_and_fetch("""
MATCH (f:Flight)-[:OPERATED_BY]->(al:Airline {name:'Lufthansa'}),
      (f)-[:DEPARTS_FROM]->(o:Airport),
      (f)-[:ARRIVES_AT]->(d:Airport)
WHERE f.departureTime < 1200
RETURN o.city AS origin, d.city AS destination, f.departureTime, f.arrivalTime
"""))
```

</details>


## Task 11: Which airports receive the most incoming flights?
<details>
<summary>Show solution</summary>

```
# 11. Which airports receive the most incoming flights?
list(db.execute_and_fetch("""
MATCH (f:Flight)-[:ARRIVES_AT]->(d:Airport)
RETURN d.city AS airport, COUNT(f) AS incomingFlights
ORDER BY incomingFlights DESC
LIMIT 5
"""))
```

</details>


## Exercise 3: Querying the Marvel Dataset in Memgraph

Start a new container named 'mcu'
```bash
docker run -d --name memgraph-marvel -p 7690:7687 memgraph/memgraph
```

Load the Marvel Dataset in Memgraph Lab

1. Open Memgraph Lab in your browser.
2. Connect to your running Memgraph instance.
3. In the sidebar, go to: Datasets --> Marvel Cinematic Universe
4. Click Load Dataset.

After loading, the database contains:

### Nodes

| Label        | Properties                             | Example |
|--------------|----------------------------------------|---------|
| `Hero`       | name (superhero name / real name)      | "SPIDER-MAN/PETER PARKER" |
|`Comic`      | name (comic title + issue/volume)      | "Astonishing Tales Vol. 2 12" |
|`ComicSeries`| title (series name), publishYear (list)| "AVENGERS VOL. 3", [2008, 2009] |


### Relationships

| Relationship          | From --> To       | Description / Example |
|----------------------|----------------|---------------------|
|`:AppearedIn`         | Hero --> Comic   | Spider-Man --> Amazing Fantasy Vol. 2 12 |
|`:AppearedInSameComic`| Hero --> Hero    | Spider-Man --> Iron Man |
|`:IsPartOfSeries`     | Comic --> ComicSeries | Amazing Fantasy --> AVENGERS VOL. 3 |

---

### Query Writing Tips
- Always use `toLower(...) CONTAINS` when searching by name. Names in the dataset are special and  stored in UPPERCASE.
Example:
```cypher
MATCH (h:Hero)
WHERE toLower(h.name) CONTAINS "thor"
RETURN h.name
LIMIT 5;
```

- Use `ANY()` for list properties.
Example: `publishYear` is a list, not a single number.
```cypher
MATCH (cs:ComicSeries)
WHERE ANY(year IN cs.publishYear WHERE year < 1980)
RETURN cs.title, cs.publishYear
LIMIT 5;
```


### Tasks

1. How many heroes/comic/comic series exist in the dataset?
2. Show the names of 10 heroes.
3. In which comics did the hero "Spider-Man" appear?
4. Which ten heroes appeared in the same comic as the hero "Iron Man"?
5. Return all comics and comic series in which the hero "Thor" appeared.
6. Find five comic series published after 1960.
7. Find ten heros who appeared in comics published after the year 2000.
8. Which heroes appeared in more than 100 comics?
9. Find 7 heros which appeared in the comic "Captain Marvel"?
10. Which comic series does the comic "Captain Marvel" belong to?


In [ ]:
# connect to memgraph 
# docker run -d --name memgraph-flights -p 7690:7687 memgraph/memgraph

from gqlalchemy import Memgraph

db = Memgraph(host="127.0.0.1", port=7690)

## Task 1: How many heros exist in the dataset?
<details>
<summary>Show solution</summary>

```
# Task 1: How many heros exist in the dataset?
list(db.execute_and_fetch("MATCH (h:Hero) RETURN count(h)"))
list(db.execute_and_fetch("MATCH (h:Hero) RETURN h.name"))
```

</details>


## Task 2: Show the names of 10 heroes.
<details>
<summary>Show solution</summary>

```
# Task 2: Show the names of 10 heroes.
list(db.execute_and_fetch("MATCH (h:Hero) RETURN h.name LIMIT 10"))
```

</details>


## Task 3: In which comics did Spider-man appear?
<details>
<summary>Show solution</summary>

```
# Task 3: In which comics did Spider-man appear?
list(db.execute_and_fetch("""
MATCH (h:Hero {name:'SPIDER-MAN/PETER PARKER'})-[:AppearedIn]->(c:Comic)
RETURN c.name;
"""))

list(db.execute_and_fetch("""
MATCH (h:Hero)-[:AppearedIn]->(c:Comic)
WHERE toLower(h.name) CONTAINS "spider-man"
RETURN c.name;
"""))
```

</details>


## Task 4: Which ten heros appeared in the same comic as Iron man?
<details>
<summary>Show solution</summary>

```
# Task 4: Which ten heros appeared in the same comic as Iron man?
list(db.execute_and_fetch("""
MATCH (h:Hero)-[:AppearedInSameComic]-(h2:Hero)
WHERE toLower(h.name) CONTAINS "iron man"
RETURN DISTINCT h2.name
LIMIT 10;"""))
```

</details>


## Task 5: Return all comics and comic series in which the hero "Thor" appeared.
<details>
<summary>Show solution</summary>

```
#Task 5: Return all comics and comic series in which the hero "Thor" appeared.
list(db.execute_and_fetch("""MATCH (h:Hero)-[:AppearedIn]->(c:Comic)-[:IsPartOfSeries]->(cs:ComicSeries)
WHERE toLower(h.name) CONTAINS "thor"
RETURN h.name, c.name AS comic, cs.title AS series;"""))
```

</details>


## Task 6: Find five comic series published after 1960.
<details>
<summary>Show solution</summary>

```
#Task 6: Find five comic series published after 1960.
list(db.execute_and_fetch("""
MATCH (cs:ComicSeries)
WHERE ANY(year IN cs.publishYear WHERE year > 1960)
RETURN cs.title, cs.publishYear
LIMIT 5;
"""))
```

</details>


## Task 7: Find 20 heros who appeared in comics published after the year 2000.
<details>
<summary>Show solution</summary>

```
# Task 7: Find 20 heros who appeared in comics published after the year 2000.
list(db.execute_and_fetch("""
MATCH (h:Hero)-[:AppearedIn]->(c:Comic)-[:IsPartOfSeries]->(cs:ComicSeries)
WHERE ANY(year IN cs.publishYear WHERE year > 2000)
RETURN DISTINCT h.name
LIMIT 20;"""))
```

</details>


## Task 7: Find 20 heros who appeared in comics published after the year 2000.
<details>
<summary>Show solution</summary>

```
# Task 7: Find 20 heros who appeared in comics published after the year 2000.
list(db.execute_and_fetch("""
MATCH (h:Hero)-[:AppearedIn]->(c:Comic)-[:IsPartOfSeries]->(cs:ComicSeries)
WHERE ANY(year IN cs.publishYear WHERE year > 2000)
RETURN DISTINCT h.name
LIMIT 20;"""))
```

</details>


## Task 7: Find 20 heros who appeared in comics published after the year 2000.
<details>
<summary>Show solution</summary>

```
# Task 7: Find 20 heros who appeared in comics published after the year 2000.
list(db.execute_and_fetch("""
MATCH (h:Hero)-[:AppearedIn]->(c:Comic)-[:IsPartOfSeries]->(cs:ComicSeries)
WHERE ANY(year IN cs.publishYear WHERE year > 2000)
RETURN DISTINCT h.name
LIMIT 20;"""))
```

</details>


## Task 7: Find 20 heros who appeared in comics published after the year 2000.
<details>
<summary>Show solution</summary>

```
# Task 7: Find 20 heros who appeared in comics published after the year 2000.
list(db.execute_and_fetch("""
MATCH (h:Hero)-[:AppearedIn]->(c:Comic)-[:IsPartOfSeries]->(cs:ComicSeries)
WHERE ANY(year IN cs.publishYear WHERE year > 2000)
RETURN DISTINCT h.name
LIMIT 20;"""))
```

</details>


## Task 8: Which heroes appeared in more than 100 comics?
<details>
<summary>Show solution</summary>

```
# Task 8: Which heroes appeared in more than 100 comics?
list(db.execute_and_fetch("""
MATCH (h:Hero)-[:AppearedIn]->(c:Comic)
WITH h, count(c) AS appearances
WHERE appearances > 100
RETURN h.name, appearances
ORDER BY appearances DESC;"""))
```

</details>


## Task 9: Find 7 heros which appeared in the comic "Captain Marvel"?
<details>
<summary>Show solution</summary>

```
# Task 9: Find 7 heros which appeared in the comic "Captain Marvel"?
list(db.execute_and_fetch("""
MATCH (h:Hero)-[:AppearedIn]->(c:Comic)
WHERE toLower(c.name) CONTAINS "marvel"
RETURN DISTINCT h.name, c.name AS comic
LIMIT 7;
"""))
```

</details>


## Task 10: Which comic series does the comic "Captain Marvel" belong to?
<details>
<summary>Show solution</summary>

```
# Task 10: Which comic series does the comic "Captain Marvel" belong to?
list(db.execute_and_fetch("""
MATCH (c:Comic)-[:IsPartOfSeries]->(cs:ComicSeries)
WHERE toLower(c.name) CONTAINS "captain marvel"
RETURN c.name AS comic, cs.title AS series;
"""))
```

</details>
